In [1]:
import scipy as sp
import numpy as np
import copy
import numpy.random as rnd

The gradient (and loss) as a function :

In [2]:
def GradERM(X, y, w, v, LambdaRegularization):
    zw = np.einsum("ij,j->i", X, w)
    zv = np.einsum("ij,j->i", X, v)
    GradwTotal = 2*np.einsum("ij,i->j", X, (zw - y)) + LambdaRegularization*w
    GradvTotal = LambdaRegularization*v
    return(np.stack((GradwTotal, GradvTotal), axis = 1))

def LossERM(X, y, w, v, LambdaRegularization):
    zw = np.einsum("ij,j->i", X, w)
    zv = np.einsum("ij,j->i", X, v)
    return( np.sum(np.power((y - zw),2), 0) + LambdaRegularization*( np.dot(w,w) + np.dot(v,v) )/2 )

The GD step

In [3]:
def GDStepERM(X, y, wv, LambdaRegularization, LearningRate):
    wv -= LearningRate*GradERM(X, y, wv[:,0], wv[:,1], LambdaRegularization)
    return(LossERM(X, y, wv[:,0], wv[:,1], LambdaRegularization))

Full GD function

In [4]:
def GDERM(X, y, wv, LambdaRegularization = 1, LearningRate = 0.02, MaxIter = 1e4, EpsConvergence = 1e-6, Verbose = True, VerboseRate = 100):
    Conv = 1
    NIter = 0
    Losses = [LossERM(X, y, wv[:,0], wv[:,1], LambdaRegularization)]
    print("Iteration %s" % NIter)
    print("Current loss %s" % Losses[NIter])
    while((NIter < MaxIter) and (Conv > EpsConvergence)):
        Losses.append(GDStepERM(X, y, wv, LambdaRegularization, LearningRate))
        NIter += 1
        Conv = np.abs(Losses[NIter] - Losses[NIter-1])/np.abs(Losses[NIter])
        if(Verbose and NIter%VerboseRate == 0):
            print("Iteration %s" % NIter)
            print("Current loss %s" % Losses[NIter])
            print("Current convergence criterion %s" % Conv)
    print("Iteration %s" % NIter)
    print("Current loss %s" % Losses[NIter])
    print("Current convergence criterion %s" % Conv)
    return(np.array(Losses))

Main variables 

In [5]:
d = 2000
LearningRate = 0.002
Reps = 20

In [6]:
for alpha in [1.0, 3.0, 5.0, 7.0, 10.0]:
    for LambdaRegularization in [1.0, 5.0]:
        m = np.zeros(2)
        m2 = np.zeros(2)
        q = np.zeros(2)
        q2 = np.zeros(2)
        for _ in range(Reps):
            M = int(alpha*d)
            X = rnd.normal(0, 1/np.sqrt(d), size = (M, d))
            wvTrue = rnd.normal(0, 1, size = (d, 2))
            wvLearned = rnd.normal(0, 1, size = (d, 2))
            yTrue = np.einsum("ij,j->i", X, wvTrue[:,0])
            alphas = M/d
            GDERM(X, yTrue, wvLearned, LearningRate=LearningRate, LambdaRegularization=LambdaRegularization, MaxIter = 100000, VerboseRate = 5000, EpsConvergence=1e-6)
            m += np.array([np.dot(wvLearned[:,0], wvTrue[:,0])/d, np.dot(wvLearned[:,1], wvTrue[:,1])/d ])
            m2 += np.square(np.array([np.dot(wvLearned[:,0], wvTrue[:,0])/d, np.dot(wvLearned[:,1], wvTrue[:,1])/d ]))
            q += np.array([np.dot(wvLearned[:,0], wvLearned[:,0])/d, np.dot(wvLearned[:,1], wvLearned[:,1])/d ])
            q2 += np.square(np.array([np.dot(wvLearned[:,0], wvLearned[:,0])/d, np.dot(wvLearned[:,1], wvLearned[:,1])/d ]))
        vm = (m2 - np.square(m)/Reps)/(Reps - 1)
        m /= Reps
        vq = (q2 - np.square(q)/Reps)/(Reps - 1)
        q /= Reps
        np.savetxt(f"q_alpha_{alpha}_Lambda_{LambdaRegularization}_d_{d}_GD_SimpleTest.txt", q, fmt="%.6f")
        np.savetxt(f"varq_alpha_{alpha}_Lambda_{LambdaRegularization}_d_{d}_GD_SimpleTest.txt", vq, fmt="%.6f")
        np.savetxt(f"m_alpha_{alpha}_Lambda_{LambdaRegularization}_d_{d}_GD_SimpleTest.txt", m, fmt="%.6f")
        np.savetxt(f"varm_alpha_{alpha}_Lambda_{LambdaRegularization}_d_{d}_GD_SimpleTest.txt", vm, fmt="%.6f")

Iteration 0
Current loss 5835.560573328616
Iteration 2266
Current loss 509.72183053195727
Current convergence criterion 9.977480152211294e-07
Iteration 0
Current loss 5740.912644790867
Iteration 2289
Current loss 488.8384375683196
Current convergence criterion 9.998149842940853e-07
Iteration 0
Current loss 6052.962983655217
Iteration 2282
Current loss 519.3441683590434
Current convergence criterion 9.985206899881135e-07
Iteration 0
Current loss 6153.402670932452
Iteration 2275
Current loss 504.95640323498185
Current convergence criterion 9.973766538157436e-07
Iteration 0
Current loss 5984.372061368332
Iteration 2280
Current loss 514.5245313774635
Current convergence criterion 9.993927670733946e-07
Iteration 0
Current loss 5837.599419516693
Iteration 2300
Current loss 484.0249428829482
Current convergence criterion 9.979038140720773e-07
Iteration 0
Current loss 5935.99262085239
Iteration 2267
Current loss 520.142352544966
Current convergence criterion 9.981556471993329e-07
Iteration 0
C